In [0]:
%run ./imports

In [0]:
dbutils.widgets.text("catalog", "user_group")
dbutils.widgets.text("schema", "project_from_zero")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

tabela = f"{catalog}.{schema}"                    # prefixo das tabelas: catalog.schema
raiz_volume = f"/Volumes/{catalog}/{schema}"

pasta_csv = f"{raiz_volume}/origem_csv"           # CSVs de cadastros (lojas, produtos, clientes)
pasta_vendas = f"{pasta_csv}/vendas"              # vendas: um arquivo novo por extração (Auto Loader)
pasta_controle = f"{raiz_volume}/controle"        # estado do Auto Loader (checkpoint e schema)

In [0]:


def log(mensagem: str):
    print(f"[{datetime.now():%H:%M:%S}] {mensagem}")


def garantir_infraestrutura():
    #Cria schema, volumes e a pasta de vendas, se ainda não existirem
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {tabela}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {tabela}.origem_csv")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {tabela}.controle")
    dbutils.fs.mkdirs(pasta_vendas)
    log(f"Ambiente pronto: {tabela}")


def _listar(pasta: str):
    try:
        return [f.name for f in dbutils.fs.ls(pasta)]
    except Exception:
        return []


def validar_arquivos(pasta: str, esperados: list):
    ##Valida existência de arquivos na pasta
    faltando = [nome for nome in esperados if nome not in _listar(pasta)]
    if faltando:
        raise FileNotFoundError(f"Nenhum arquivo encontrado em {pasta}: {faltando}")
    log(f"Arquivos conferidos em {pasta}: {esperados}")


def validar_pasta_com_csv(pasta: str):
    csvs = [nome for nome in _listar(pasta) if nome.lower().endswith(".csv")]
    if not csvs:
        raise FileNotFoundError(f"Nenhum CSV encontrado em {pasta}.")
    log(f"{len(csvs)} arquivo(s) CSV em {pasta}")
    return csvs


def adicionar_metadados(df, origem: str):
    ##Colunas de rastreabilidade da Bronze
    return df.withColumn("_ingestao_ts", F.current_timestamp()).withColumn("_origem", F.lit(origem))


def resetar_bronze_vendas():
    ## apaga bronze_vendas e o estado do Auto Loader, para reprocessar todos os arquivos do zero
    spark.sql(f"DROP TABLE IF EXISTS {tabela}.bronze_vendas")
    dbutils.fs.rm(f"{pasta_controle}/bronze_vendas", True)
    log("bronze_vendas e checkpoint apagados")




In [0]:
log(f"Utilitários carregados · destino: {tabela}")